# Module 9: Cheat Sheet

| Section | Topic |
|---------|-------|
| 1 | Hyperparameters |
| 2 | Text Serialisation |
| 3 | Tokenisers |
| 4 | `TextDataset` Class |
| 5 | `create_dataloaders` Helper |
| 6 | The Four Transformer Models |
| 7 | Training: Loss & Optimiser |
| 8 | Evaluation: Metrics & Functions |


---
## 1 · Hyperparameters

```python
MAX_LENGTH = 64    # starter (diabetes text is short, ~40-55 tokens)
BATCH_SIZE = 16    # mini-batch size fed to the model at each step
NUM_EPOCHS = 3     # number of full passes through the training set
LR         = 2e-5  # learning rate -> much smaller than Week 3's 0.001
```
> BATCH_SIZE = 16 means the model sees 16 samples at once before updating its weights.
Instead of processing all training samples in one go, the dataset is split into chunks (batches) of 16. After each batch, the loss is computed and the optimiser takes one step.

Training set: 614 samples, BATCH_SIZE=16
```
Epoch 1:
  Batch 1 → samples 1–16   → compute loss → update weights
  Batch 2 → samples 17–32  → compute loss → update weights
  ...
  Batch 39 → samples 609–614 → compute loss → update weights
  ← one full epoch done (≈ 39 steps)
```
> number of batches = ⌈ training samples / BATCH_SIZE ⌉ 
> The ceiling function ⌈x⌉ means "round up to the nearest integer"

### Why these values?

| Parameter | Week 3 | Week 4 | Reason |
|-----------|--------|--------|--------|
| Epochs | 50 | 3 | Pre-trained weights need only small adjustments; more epochs → overfitting |
| LR | 0.001 | 2e-5 | Large LR destroys pre-trained representations ("catastrophic forgetting") |
| Optimiser | `Adam` | `AdamW` | AdamW decouples weight decay from the adaptive LR update; better regularisation |
| Loss | `BCEWithLogitsLoss` | `CrossEntropyLoss` | Transformers output **two logits** (one per class), not one |

### MAX\_LENGTH rule of thumb

```
Serialised diabetes row  ≈ 40–55 tokens  →  MAX_LENGTH = 64
Natural language sentence ≈ 20–40 tokens →  MAX_LENGTH = 128 (BERT default)
```

> Tokens beyond `MAX_LENGTH` are **truncated** and lost.
> Sequences shorter than `MAX_LENGTH` are **padded** to the same length.


---
## 2 · Text Serialisation

### Concept

Transformer models (BERT, GPT-2, …) were pre-trained on **text**.
To apply them to a tabular dataset, each row must be converted into a string
that the tokeniser can process.

### Pattern

```
"Feature1: value1, Feature2: value2, ..., FeatureN: valueN"
```

### Examples

**Diabetes (starter)**
```
"Pregnancies: 6, Glucose: 148, BloodPressure: 72, SkinThickness: 35,
 Insulin: 0, BMI: 33.60, DiabetesPedigreeFunction: 0.63, Age: 50"
```

**Body Performance (exercises)**
```
"age: 30, gender: M, height_cm: 175.00, weight_kg: 70.50, body fat %: 20.00,
 diastolic: 80, systolic: 120, gripForce: 40.00, ..."
```


---
## 3 · Tokenisers

### What a tokeniser does?

```
Raw text string
       │
       ▼  tokeniser(text, truncation=True, padding=True, max_length=64)
       │
       ├── input_ids       : list[int]  -> integer ID for each token
       ├── attention_mask  : list[int]  -> 1 for real tokens, 0 for padding
       └── token_type_ids  : list[int]  -> segment IDs (BERT only; 0 or 1)
```

### Tokeniser comparison

| Model | Tokeniser type | Vocabulary | Special tokens | Padding token |
|-------|---------------|------------|----------------|---------------|
| BERT | WordPiece | 30 522 | `[CLS]` … `[SEP]` | `[PAD]` (id 0) |
| GPT-2 | Byte-level BPE | 50 257 | None / `<|endoftext|>` | ⚠️ **none** -> must be set manually |
| RoBERTa | Byte-level BPE | 50 265 | `<s>` … `</s>` | `<pad>` |
| DistilBERT | WordPiece (same as BERT) | 30 522 | `[CLS]` … `[SEP]` | `[PAD]` (id 0) |

### Loading a tokeniser

```python
from transformers import AutoTokenizer

# AutoTokenizer picks the correct class automatically
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
```

### The GPT-2 padding fix 

GPT-2 was designed for text generation (no padding needed).
Classification requires fixed-length batches → padding is required.
We reuse the end-of-sequence token `<|endoftext|>` as the pad token:

```python
# Step 1: tell the tokeniser which token to use for padding
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

# Step 2: tell the model the same pad_token_id so it ignores padding positions
gpt2_model.config.pad_token_id = gpt2_tokenizer.eos_token_id
```

> If you skip Step 2, GPT-2 may attend to padding positions and produce
> incorrect classifications.

### Tokenising a list of strings

```python
encodings = tokenizer(
    list_of_strings,      # list[str]
    truncation=True,      # cut sequences longer than max_length
    padding=True,         # pad shorter sequences to the longest in the batch
    max_length=64,        # hard cap
)
# encodings is a BatchEncoding dict:
#   encodings['input_ids']      → list of lists
#   encodings['attention_mask'] → list of lists
```


---
## 4 · `TextDataset` Class

### Purpose

Bridges the gap between HuggingFace tokeniser output (a plain Python dict)
and PyTorch's `DataLoader` (which needs an object with `__len__` and `__getitem__`).

### Code

```python
from torch.utils.data import Dataset
import torch

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings  # BatchEncoding dict from tokeniser
        self.labels    = labels     # list or array of integer class labels

    def __len__(self):
        return len(self.labels)     # number of samples

    def __getitem__(self, idx):
        # Convert each encoding sequence at position idx to a tensor
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        # Add the label as a long (int64) tensor -> required by CrossEntropyLoss
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
```

### Anatomy of one item

```python
sample = dataset[0]
# sample is a dict:
# {
#   'input_ids'     : tensor([101, 8795, 1024, 6, ...])  shape (MAX_LENGTH,)
#   'attention_mask': tensor([1, 1, 1, ..., 0, 0])       shape (MAX_LENGTH,)
#   'labels'        : tensor(1)                           scalar long
# }
```

### Comparison with Week 3

| | Week 3 | Week 4 |
|-|--------|--------|
| Dataset class | `TensorDataset(X_tensor, y_tensor)` | `TextDataset(encodings, labels)` |
| Input type | Pre-built float tensors | Dict of integer tensors from tokeniser |
| Label dtype | `torch.float32` (BCEWithLogitsLoss) | `torch.long` (CrossEntropyLoss) |


---
## 5 · `create_dataloaders` Helper

### Purpose

Wraps tokenisation + `TextDataset` construction + `DataLoader` creation
into one reusable function -> called once per model (each model has its own tokeniser).

### Code

```python
from torch.utils.data import DataLoader

def create_dataloaders(tokenizer, train_texts, y_train,
                       test_texts,  y_test,
                       max_length=MAX_LENGTH, batch_size=BATCH_SIZE):

    # 1. Tokenise both splits
    train_enc = tokenizer(train_texts, truncation=True,
                          padding=True, max_length=max_length)
    test_enc  = tokenizer(test_texts,  truncation=True,
                          padding=True, max_length=max_length)

    # 2. Wrap in TextDataset
    train_ds = TextDataset(train_enc, list(y_train))
    test_ds  = TextDataset(test_enc,  list(y_test))

    # 3. Create DataLoaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

    return train_loader, test_loader
```

### Key arguments

| Argument | Value | Reason |
|----------|-------|--------|
| `truncation=True` | Always | Prevents crash on sequences > `max_length` |
| `padding=True` | Always | Ensures all sequences in a batch are the same length |
| `shuffle=True` | Train only | Randomises batch order → better gradient estimates |
| `shuffle=False` | Test only | Preserves order so labels align with predictions |

### Usage pattern

```python
# Called once per model -> each tokeniser produces a separate pair of loaders
bert_train_loader, bert_test_loader = create_dataloaders(
    bert_tokenizer, train_texts, y_train, test_texts, y_test
)
gpt2_train_loader, gpt2_test_loader = create_dataloaders(
    gpt2_tokenizer, train_texts, y_train, test_texts, y_test
)
# … and so on for RoBERTa and DistilBERT
```

### What a batch looks like

```python
batch = next(iter(bert_train_loader))
# batch is a dict:
# {
#   'input_ids'     : tensor shape (BATCH_SIZE, MAX_LENGTH)  — int64
#   'attention_mask': tensor shape (BATCH_SIZE, MAX_LENGTH)  -> int64
#   'labels'        : tensor shape (BATCH_SIZE,)             -> int64
# }
```


---
## 6 · The Four Transformer Models

### Quick Reference

| Model | HuggingFace ID | Type | Layers | Params | Pre-training objective | Padding token |
|-------|---------------|------|--------|--------|----------------------|---------------|
| BERT | `bert-base-uncased` | Encoder | 12 | ~110 M | MLM + NSP | `[PAD]` |
| GPT-2 | `gpt2` | Decoder | 12 | ~117 M | Causal LM | none -> set manually |
| RoBERTa | `roberta-base` | Encoder | 12 | ~125 M | MLM (no NSP, more data) | `<pad>` |
| DistilBERT | `distilbert-base-uncased` | Encoder | 6 | ~66 M | Knowledge distillation | `[PAD]`|

### Architecture: Encoder vs Decoder

```
ENCODER (BERT, RoBERTa, DistilBERT)
  Input → [CLS] token1 token2 … [SEP]
  Each token attends to ALL other tokens (bidirectional)
  Classification head sits on the [CLS] representation

DECODER (GPT-2)
  Input → token1 token2 … tokenN
  Each token attends only to PREVIOUS tokens (causal / unidirectional)
  Classification head sits on the LAST non-padding token representation
```

### Loading a model for classification

```python
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,       # 2 for binary, 3+ for multiclass
).to(device)
```

`AutoModelForSequenceClassification` automatically:
1. Loads the pre-trained transformer weights.
2. Adds a **randomly initialised linear head** on top: `Linear(hidden_size, num_labels)`.
3. Fine-tuning trains **all** weights (transformer + head) jointly.

### What the model outputs

```python
outputs = model(input_ids=input_ids, attention_mask=attention_mask)

outputs.logits  # raw scores, shape (batch_size, num_labels)
                # NOT probabilities -> apply softmax to get those
```

### DistilBERT: knowledge distillation in brief

```
BERT (teacher, 12 layers)  →  trains  →  DistilBERT (student, 6 layers)
                            loss = cross-entropy on soft teacher probabilities
                                 + cosine distance between hidden states
Result: 40% fewer params, ~60% faster, ~97% of BERT's GLUE performance
```


---
## 7 · Training: Loss Function & Optimiser

### CrossEntropyLoss vs BCEWithLogitsLoss

| | `BCEWithLogitsLoss` | `CrossEntropyLoss` |
|-|--------------------|-----------------|
| Model output shape | `(N, 1)` -> one logit | `(N, C)` -> one logit per class |
| Label dtype | `torch.float32` | `torch.long` |
| Internal operation | sigmoid → binary cross-entropy | softmax → categorical cross-entropy |
| Works for 2+ classes? | Binary only | Any C ≥ 2 |
```python
# Week 4 loss
criterion = nn.CrossEntropyLoss()
loss = criterion(outputs.logits, labels)  # logits: (B, C), labels: (B,) long
```

### **CrossEntropyLoss works for binary classification too**

```
CrossEntropyLoss with C=2:
  logits  = [z₀, z₁]  (raw scores for class 0 and class 1)
  softmax → [p₀, p₁]  where p₀ + p₁ = 1
  loss    = -log(p_{true_class})

This is mathematically equivalent to BCE when C=2,
**but the model outputs 2 logits instead of 1**.
```

### AdamW optimiser

```python
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,           # fine-tuning standard: 1e-5 to 5e-5
    weight_decay=0.01  # L2 regularisation applied correctly (decoupled)
)
```

**Adam vs AdamW:**
Regular Adam mixes the weight-decay gradient into the adaptive update,
which weakens regularisation. AdamW applies weight decay directly to the weights,
independently of the gradient -> better regularisation for large pre-trained models.

### The training loop

```python
model.train()

for epoch in range(NUM_EPOCHS):
    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()                                      # clear old gradients
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask)             # forward pass
        loss = criterion(outputs.logits, labels)                   # compute loss
        loss.backward()                                            # backprop
        optimizer.step()                                           # update weights
```


---
## 8 · Evaluation: Metrics & Functions

### From logits to predictions

```python
# After collecting all logits across batches:
all_logits = torch.cat(all_logits, dim=0)   # shape: (N, C)

probabilities = torch.softmax(all_logits, dim=1).numpy()  # shape: (N, C)
predictions   = torch.argmax(all_logits, dim=1).numpy()   # shape: (N,) -> class labels
```

### Sigmoid vs Softmax

| | `sigmoid` | `softmax` |
|-|-----------|----------|
| Input shape | `(N, 1)` | `(N, C)` |
| Output | probability in (0, 1) for ONE class | probability distribution over ALL C classes |
| Outputs sum to 1? | independent | mutually exclusive |

### Binary classification metrics (starter)

```python
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy_score (y_test, predictions)                   # fraction correct
precision_score(y_test, predictions, zero_division=0)  # default average='binary'
recall_score   (y_test, predictions, zero_division=0)
f1_score       (y_test, predictions, zero_division=0)

# ROC-AUC for binary: pass probability of the POSITIVE class only
roc_auc_score(y_test, probabilities[:, 1])             # column 1 = P(class=1)
```

### Multiclass metrics (exercises)

```python
precision_score(y_test, predictions, average='weighted', zero_division=0)
recall_score   (y_test, predictions, average='weighted', zero_division=0)
f1_score       (y_test, predictions, average='weighted', zero_division=0)

# ROC-AUC for multiclass: pass the FULL probability matrix
roc_auc_score(y_test, probabilities, multi_class='ovr', average='weighted')
```
> `multi_class='ovr'` will be explained in the lecture

| Value | Meaning | When to use |
|-------|---------|-------------|
| `'binary'` | Score for the positive class only | 2-class problems |
| `'macro'` | Unweighted mean across all classes | All classes equally important |
| `'weighted'` | Mean weighted by number of samples per class | Imbalanced class sizes |
| `'micro'` | Global sum of TP / FP / FN across all classes | multi-class and multi-label settings |

> `average` parameter will be explained in the lecture

```
One-vs-Rest (OvR) strategy for 4-class ROC-AUC:

  Class A: is it A or not-A?  → compute AUC_A using P(class=A) as the score
  Class B: is it B or not-B?  → compute AUC_B using P(class=B)
  Class C: is it C or not-C?  → compute AUC_C using P(class=C)
  Class D: is it D or not-D?  → compute AUC_D using P(class=D)

  Final ROC_AUC = weighted average of AUC_A, AUC_B, AUC_C, AUC_D
```

### Confusion matrix

```python
from sklearn.metrics import ConfusionMatrixDisplay

# Binary (starter)
ConfusionMatrixDisplay.from_predictions(
    all_labels, all_preds,
    display_labels=['No Diabetes', 'Diabetes'],
    colorbar=False, ax=ax
)

# 4-class (exercises)
ConfusionMatrixDisplay.from_predictions(
    all_labels, all_preds,
    display_labels=['A', 'B', 'C', 'D'],
    colorbar=False, ax=ax
)
```

### Why evaluate_model collects all batches first

Transformer evaluation **cannot** be done in a single call like Week 3's
`model(X_test.to(device))` because:
1. Text must be tokenised into integer tensors beforehand.
2. Large test sets would exceed GPU memory in a single batch.

```python
# Pattern: accumulate → concatenate → compute metrics
all_logits, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = model(input_ids=..., attention_mask=...)
        all_logits.append(outputs.logits.cpu())
        all_labels.append(batch['labels'].cpu())
all_logits = torch.cat(all_logits, dim=0)  # then compute metrics once
```
---
